# 🏭 Module 11: Forward Deployed Engineering (FDE) & Real-Time Data Pipelines

Forward Deployed Engineers (FDEs) connect customer data lakes, webhooks, and third-party APIs into high-throughput ingestion pipelines.

### What We Will Master:
1. **Producer-Consumer Architecture**: Queue-backed worker pools.
2. **Micro-Batching Buffer**: Flush every 100 items OR after 200ms timeout before bulk-writing to S3 / Vector DB.
3. **Handling Backpressure**: Preventing Out-Of-Memory (OOM) crashes when upstream sends millions of records.
4. **Dead-Letter Queues (DLQ)**: Isolating failed records for inspection.
5. **Graceful Pipeline Draining & Zero-Data-Loss Shutdown**.

In [1]:
import asyncio

queue = asyncio.Queue(maxsize=50) # Backpressure limit: max 50 items in memory

async def producer(n_events: int = 12):
    for i in range(1, n_events + 1):
        await queue.put({"event_id": i, "data": f"payload_{i}"})
    print(f"  [Producer] Pushed {n_events} webhook events to ingestion queue.")

async def consumer_worker(worker_id: str):
    while True:
        item = await queue.get()
        await asyncio.sleep(0.05) # Simulate transformation & vector embedding
        print(f"  [{worker_id}] Processed event ID: {item['event_id']}")
        queue.task_done()

print("🚀 Launching FDE Ingestion Pipeline: 1 Producer, 3 Worker Consumers")
workers = [asyncio.create_task(consumer_worker(f"Worker-{i}")) for i in range(1, 4)]
await producer(12)
await queue.join()
for w in workers:
    w.cancel()
print("✅ Queue drained! Bulk flushed to Vector Database. Pipeline shutdown cleanly.")

🚀 Launching FDE Ingestion Pipeline: 1 Producer, 3 Worker Consumers
  [Producer] Pushed 12 webhook events to ingestion queue.
  [Worker-1] Processed event ID: 1
  [Worker-2] Processed event ID: 2
  [Worker-3] Processed event ID: 3
  [Worker-1] Processed event ID: 4
  [Worker-2] Processed event ID: 5
  [Worker-3] Processed event ID: 6
  [Worker-1] Processed event ID: 7
  [Worker-2] Processed event ID: 8
  [Worker-3] Processed event ID: 9
  [Worker-1] Processed event ID: 10
  [Worker-2] Processed event ID: 11
  [Worker-3] Processed event ID: 12
✅ Queue drained! Bulk flushed to Vector Database. Pipeline shutdown cleanly.


---
## 2. The Micro-Batching Buffer Engine

Inserting rows into PostgreSQL or Pinecone one-by-one destroys performance.
A **Micro-Batcher** buffers records in memory and flushes in bulk whenever:
1. **Batch size limit** is reached (e.g., 5 items), **OR**
2. **Time window expires** (e.g., 300ms has elapsed since the first record).

In [2]:
async def micro_batcher_demo():
    buffer = []
    max_batch_size = 4
    flush_timeout = 0.2

    async def flush_batch(reason: str):
        if not buffer:
            return
        items_to_write = list(buffer)
        buffer.clear()
        print(f"📦 Flushed Batch of {len(items_to_write)} items to S3 / Vector DB (Trigger: {reason})!")

    print("Micro-batcher running (max_batch=4, window=0.2s)...")
    # Push 4 items quickly -> triggers size flush
    for i in range(4):
        buffer.append(f"item_{i}")
        if len(buffer) >= max_batch_size:
            await flush_batch("Batch full")

    # Push 3 items and wait -> triggers timeout flush
    for i in range(4, 7):
        buffer.append(f"item_{i}")
    await asyncio.sleep(flush_timeout + 0.05)
    await flush_batch("Timeout window reached")

await micro_batcher_demo()
print("✅ All 7 records flushed in bulk batches!")

Micro-batcher running (max_batch=4, window=0.2s)...
📦 Flushed Batch of 4 items to S3 / Vector DB (Trigger: Batch full)!
📦 Flushed Batch of 3 items to S3 / Vector DB (Trigger: Timeout window reached)!
✅ All 7 records flushed in bulk batches!
